In [2]:
import numpy as np
import pandas as pd
np.random.seed(42)

In [3]:
X=np.random.uniform(-2,2,(400,3))

y=np.sin(X[:,0])+0.5*(X[:,1]**2)-0.8*X[:,2]
y=y.reshape(-1,1)

X=X.T
y=y.T

In [4]:
#step-2:defining the activation functions and their derivatives
def relu(Z):
    return np.maximum(0,Z)

def drelu(Z):
    return (Z>0).astype(float)


def sigmoid(Z):
    return 1/(1+np.exp(-Z))

def dsigmoid(Z):
    s=sigmoid(Z)
    return s*(1-s)


def tanh(Z):
    return np.tanh(Z)

def dtanh(Z):
    return 1-np.tanh(Z)**2


def leaky_relu(Z,alpha=0.01):
    return np.where(Z>0,Z,alpha*Z)

def dleaky_relu(Z,alpha=0.01):
    return np.where(Z>0,1,alpha)


def softplus(Z):
    return np.log(1+np.exp(Z))

def dsoftplus(Z):
    return 1/(1+np.exp(-Z))

In [5]:
#step 3: initalizing the parameters
def initialize(layer_dims):
    parameters={}
    L=len(layer_dims)-1

    for l in range(1,L+1):
        parameters["W"+str(l)]=np.random.uniform(-0.5,0.5,(layer_dims[l],layer_dims[l-1]))
        parameters["b"+str(l)]=np.zeros((layer_dims[l],1))

    return parameters

In [6]:
#step 4: generalized forwad propagaion
def forward(X,parameters,activation):
    caches=[]
    A=X
    L=len(parameters)//2

    for l in range(1,L):
        W=parameters["W"+str(l)]
        b=parameters["b"+str(l)]

        Z=W@A+b
        A=activation(Z)

        caches.append((A,Z))

    W=parameters["W"+str(L)]
    b=parameters["b"+str(L)]

    Z=W@A+b
    A=Z

    caches.append((A,Z))

    return A,caches

In [7]:
#step 5: generalized backward propagation
def backward(X,y,parameters,caches,activation_derivative):
    grads={}
    L=len(parameters)//2
    m=X.shape[1]

    A_final=caches[-1][0]
    dA=2*(A_final-y)

    for l in reversed(range(1,L+1)):

        A,Z=caches[l-1]

        if l==L:
            dZ=dA
        else:
            dZ=dA*activation_derivative(Z)

        if l==1:
            A_prev=X
        else:
            A_prev=caches[l-2][0]

        grads["dW"+str(l)]=(1/m)*(dZ@A_prev.T)
        grads["db"+str(l)]=(1/m)*np.sum(dZ,axis=1,keepdims=True)

        dA=parameters["W"+str(l)].T@dZ

    return grads

In [8]:
#step 6: upadating the gradient descents
def update(parameters,grads,lr):
    L=len(parameters)//2

    for l in range(1,L+1):
        parameters["W"+str(l)]-=lr*grads["dW"+str(l)]
        parameters["b"+str(l)]-=lr*grads["db"+str(l)]

    return parameters

In [9]:
#step 7:training
def train(layer_dims,activation,activation_derivative):

    parameters=initialize(layer_dims)

    lr=0.01
    epochs=1000
    loss_200=None

    for epoch in range(epochs):

        y_hat,caches=forward(X,parameters,activation)
        loss=np.mean((y_hat-y)**2)

        if epoch==200:
            loss_200=loss

        grads=backward(X,y,parameters,caches,activation_derivative)
        parameters=update(parameters,grads,lr)

    grad_norm_L1=np.linalg.norm(grads["dW1"])
    last_hidden=len(layer_dims)-2
    grad_norm_last=np.linalg.norm(grads["dW"+str(last_hidden)])

    return loss,loss_200,grad_norm_L1,grad_norm_last

In [10]:
models={
    "Model A (Shallow-1HiddenLayer)":[3,4,1],
    "Model B (Medium-2HiddenLayers)":[3,6,6,1],
    "Model C (Deep-4HiddenLayers)":[3,8,8,8,8,1],
    "Model D (VeryDeep-8HiddenLayers)":[3,8,8,8,8,8,8,8,8,1]
}

In [11]:
results=[]

for name in [
    "Model A (Shallow-1HiddenLayer)",
    "Model B (Medium-2HiddenLayers)",
    "Model C (Deep-4HiddenLayers)"
]:
    dims=models[name]
    final_loss,loss_200,g1,glast=train(dims,relu,drelu)
    results.append([name,"ReLU",final_loss,loss_200,g1,glast])

name="Model D (VeryDeep-8HiddenLayers)"

final_loss,loss_200,g1,glast=train(models[name],relu,drelu)
results.append([name,"ReLU",final_loss,loss_200,g1,glast])

final_loss,loss_200,g1,glast=train(models[name],sigmoid,dsigmoid)
results.append([name,"Sigmoid",final_loss,loss_200,g1,glast])

In [12]:
df=pd.DataFrame(
    results,
    columns=[
        "Model",
        "Activation",
        "FinalLoss",
        "Loss@200",
        "GradNorm_L1",
        "GradNorm_LastHidden"
    ]
)

print(df)

                              Model Activation  FinalLoss  Loss@200  \
0    Model A (Shallow-1HiddenLayer)       ReLU   0.111545  0.492686   
1    Model B (Medium-2HiddenLayers)       ReLU   0.072886  0.320594   
2      Model C (Deep-4HiddenLayers)       ReLU   0.030451  0.847796   
3  Model D (VeryDeep-8HiddenLayers)       ReLU   0.053185  1.632672   
4  Model D (VeryDeep-8HiddenLayers)    Sigmoid   1.743850  1.743850   

   GradNorm_L1  GradNorm_LastHidden  
0     0.045217             0.045217  
1     0.036609             0.021441  
2     0.023876             0.016801  
3     0.429784             0.621290  
4     0.000006             0.000006  


Reflections

1.Did deeper always reduce loss faster?

Deeper networks did not always reduce the loss faster. The performance improved as the model depth increased up to 4 hidden layers, but increasing the depth further did not lead to better results. In fact, when sigmoid activation was used in the very deep model, the loss increased significantly.

2.Did gradients in early layers stay similar to later layers?

The gradients in early layers did not remain similar to those in later layers as the network became deeper. In the very deep sigmoid model, the gradient norms were almost zero, which indicates vanishing gradients and explains the poor learning.


3.Was training equally stable for all activations?

Training was not equally stable for all activation functions. Models using ReLU showed stable and consistent learning, while the very deep model with sigmoid struggled due to extremely small gradients.



4.Which activation behaved more stable in deeper networks?

ReLU behaved more stably in deeper networks, maintaining meaningful gradient magnitudes and achieving lower loss compared to sigmoid.



5.Did some models improve very slowly even though learning rate was same?

Even though the learning rate was the same for all models, some models improved very slowly. The very deep sigmoid model, in particular, had very small gradient norms, resulting in minimal parameter updates and slow improvement.